# Cost-aware Stopping for MOEA


## Packages

In [ ]:
pip install pymoo==0.6.1.6 platypus-opt numpy scipy pandas

In [2]:
from pathlib import Path
from collections import Counter, deque
from decimal import Decimal, ROUND_HALF_UP

import math
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from pymoo.algorithms.moo.moead import MOEAD
from pymoo.problems import get_problem
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from platypus import EpsilonBoxArchive, Problem as PlatypusProblem, Solution

## Experiment

In [ ]:
# Experiment settings
PROBLEMS = [f"dtlz{i}" for i in range(1, 8)]
M_LIST = [2, 3, 5]

POP_SIZE = 50 
FE_MAX = 10000
N_RUNS = 31
SEED0 = 2026

MOEAD_REF_SEED = 1

R2_N_WEIGHTS = 1000
R2_SEED = 12345

RHO = 10.0
C_FE = RHO / FE_MAX

# Proposed stopping rule
T_MIN = 20
PATIENCE = 5

# MGBM
MGBM_R = 0.05
MGBM_THRESHOLD = 0.12

# Entropy-based stopping criterion
ESC_WINDOW = 20
ESC_BINS = 10
ESC_ROUND = "0.001"

# Epsilon-progress stopping criterion
EPSILON = 0.2
EPS_STAGNATION = 10

OUT = Path("moead_stopping_results.csv")

In [4]:
# DTLZ problems and normalization
DTLZ_K = {
    "dtlz1": 5,
    "dtlz2": 10,
    "dtlz3": 10,
    "dtlz4": 10,
    "dtlz5": 10,
    "dtlz6": 10,
    "dtlz7": 20,
}

DTLZ7_XMAX = 0.859400856226174
DTLZ7_HMAX = DTLZ7_XMAX * (1 + np.sin(3 * np.pi * DTLZ7_XMAX))

_NDS = NonDominatedSorting()
_REF_CACHE = {}
_R2_CACHE = {}

def make_problem(name, m):
    n_var = m - 1 + DTLZ_K[name]
    return get_problem(name, n_var=n_var, n_obj=m)


def true_ideal_nadir(name, m):
    ideal = np.zeros(m)

    if name == "dtlz1":
        return ideal, np.full(m, 0.5)

    if name in {"dtlz2", "dtlz3", "dtlz4"}:
        return ideal, np.ones(m)

    if name in {"dtlz5", "dtlz6"}:
        q = 1 / np.sqrt(2)
        nadir = np.ones(m)
        nadir[0] = q ** (m - 2)

        for j in range(1, m - 1):
            nadir[j] = q ** (m - j - 1)

        return ideal, nadir

    if name == "dtlz7":
        ideal[-1] = 2 * m - (m - 1) * DTLZ7_HMAX
        nadir = np.full(m, DTLZ7_XMAX)
        nadir[-1] = 2 * m
        return ideal, nadir

    raise ValueError(f"Unknown problem: {name}")


def normalize(F, ideal, nadir):
    F = np.asarray(F, dtype=float)
    return (F - ideal) / (nadir - ideal)

In [5]:
# generation buffer + trajectory saving
TRAJECTORY_DIR = Path("moead_trajectories")
TRAJECTORY_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# MOEA/D reference directions and R2 weights
def moead_ref_dirs(m):
    if m not in _REF_CACHE:
        _REF_CACHE[m] = get_reference_directions(
            "energy", m, POP_SIZE, seed=MOEAD_REF_SEED
        )
    return _REF_CACHE[m].copy()

# R2 weights with 1000 weights
def r2_weights(m):
    if m not in _R2_CACHE:
        rng = np.random.default_rng(R2_SEED + m)
        _R2_CACHE[m] = rng.dirichlet(np.ones(m), R2_N_WEIGHTS)
    return _R2_CACHE[m]

In [7]:
# External nondominated archive: best nondominated solutions discovered so far.
def nd_only(F):
    F = np.unique(np.asarray(F, dtype=float), axis=0)

    if len(F) <= 1:
        return F

    idx = _NDS.do(F, only_non_dominated_front=True)
    return F[idx]


def update_archive(A, B):
    B = nd_only(B)

    if len(A) == 0:
        return B

    le = A[:, None, :] <= B[None, :, :]
    lt = A[:, None, :] < B[None, :, :]

    b_dominated = (le.all(axis=2) & lt.any(axis=2)).any(axis=0)
    b_duplicate = np.equal(A[:, None, :], B[None, :, :]).all(axis=2).any(axis=0)

    B = B[~(b_dominated | b_duplicate)]

    if len(B) == 0:
        return A

    le = B[:, None, :] <= A[None, :, :]
    lt = B[:, None, :] < A[None, :, :]
    a_dominated = (le.all(axis=2) & lt.any(axis=2)).any(axis=0)

    return np.vstack((A[~a_dominated], B))

In [8]:
# R2 indicator
def r2_min_by_weight(F_norm, weights):
    scalar_values = np.max(F_norm[:, None, :] * weights[None, :, :], axis=2)
    return scalar_values.min(axis=0)

In [ ]:
# Learning-curve prediction
def learning_curve_models():
    return [
        (
            lambda s, p: p[0] + p[1] * np.exp(-p[2] * s),
            [0, 0, 1e-8],
            [np.inf, np.inf, 10.0],
        ),
        (
            lambda s, p: p[0] + p[1] * (s + p[2]) ** (-p[3]),
            [0, 0, 1e-3, 0.05],
            [np.inf, np.inf, 1000.0, 10.0],
        ),
        (
            lambda s, p: p[0] + p[1] * np.exp(-(p[2] * s) ** p[3]),
            [0, 0, 1e-8, 0.1],
            [np.inf, np.inf, 10.0, 5.0],
        ),
        (
            lambda s, p: p[0] + p[1] / (1 + p[2] * s) ** p[3],
            [0, 0, 1e-8, 0.05],
            [np.inf, np.inf, 10.0, 10.0],
        ),
    ]


def fit_predict_next(y):
    y = np.asarray(y, dtype=float)
    s = np.arange(len(y), dtype=float)
    s_next = float(len(y))
    current = float(y[-1])

    asymptote0 = max(0, min(y[-5:]) * 0.95)
    amplitude0 = max(y[0] - asymptote0, 1e-8)

    initial_params = [
        [asymptote0, amplitude0, 0.05],
        [asymptote0, amplitude0, 1.0, 0.5],
        [asymptote0, amplitude0, 0.05, 1.0],
        [asymptote0, amplitude0, 0.05, 1.0],
    ]

    fitted_models = []

    for (model, lower, upper), p0 in zip(learning_curve_models(), initial_params):
        upper = np.asarray(upper, dtype=float)
        upper[:2] = max(float(np.max(y)) * 10 + 1, 1.0)

        constraint = {
            "type": "ineq",
            "fun": lambda p, f=model: current - f(s_next, p),
        }                                                                ## Elitist prediction condition

        result = minimize(
            lambda p, f=model: np.mean((f(s, p) - y) ** 2),
            p0,
            method="SLSQP",
            bounds=list(zip(lower, upper)),
            constraints=constraint,
            options={"maxiter": 1500, "ftol": 1e-12},
        )

        if result.success and np.isfinite(result.fun):
            fitted_models.append((model, result.x, result.fun))

    if not fitted_models:
        return None

    fitted_history = np.column_stack(
        [model(s, params) for model, params, _ in fitted_models]
    )

    next_predictions = np.array(
        [model(s_next, params) for model, params, _ in fitted_models]
    )

    n_models = len(fitted_models)
    weights0 = np.full(n_models, 1 / n_models)

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "ineq", "fun": lambda w: current - next_predictions @ w},   ## Elitist prediction condition
    ]

    result = minimize(
        lambda w: np.mean((fitted_history @ w - y) ** 2),
        weights0,
        method="SLSQP",
        bounds=[(0, 1)] * n_models,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-12},
    )

    if result.success:
        return float(next_predictions @ result.x)

    errors = [error for _, _, error in fitted_models]
    best_idx = int(np.argmin(errors))
    return float(next_predictions[best_idx])

In [ ]:
# run trajectory
def run_trajectory(name, m, seed):
    problem = make_problem(name, m)
    ideal, nadir = true_ideal_nadir(name, m)
    weights = r2_weights(m)

    algorithm = MOEAD(ref_dirs=moead_ref_dirs(m))
    algorithm.setup(
        problem,
        termination=("n_eval", FE_MAX),
        seed=seed,
        verbose=False,
    )

    # Initial population: 100 FEs
    algorithm.next()

    F0 = np.asarray(algorithm.pop.get("F"), dtype=float).copy()
    archive = nd_only(F0)

    F0_norm = normalize(F0, ideal, nadir)
    best_by_weight = r2_min_by_weight(F0_norm, weights)

    populations = [F0]
    offspring = []
    fes = [int(algorithm.evaluator.n_eval)]
    r2_values = [float(best_by_weight.mean())]
    archive_sizes = [len(archive)]

    offspring_buffer = []

    # pymoo MOEA/D evaluates one offspring at a time.
    # We record one checkpoint after every POP_SIZE offspring.
    while algorithm.evaluator.n_eval < FE_MAX:
        before = int(algorithm.evaluator.n_eval)
        algorithm.next()
        after = int(algorithm.evaluator.n_eval)

        if after == before:
            continue

        F_new = np.asarray(algorithm.off.get("F"), dtype=float).reshape(-1, m)

        for f in F_new:
            offspring_buffer.append(f.copy())

        if len(offspring_buffer) == POP_SIZE:
            Q = np.asarray(offspring_buffer, dtype=float)
            offspring_buffer = []

            offspring.append(Q)

            archive = update_archive(archive, Q)

            Q_norm = normalize(Q, ideal, nadir)
            new_best = r2_min_by_weight(Q_norm, weights)
            best_by_weight = np.minimum(best_by_weight, new_best)

            populations.append(
                np.asarray(algorithm.pop.get("F"), dtype=float).copy()
            )

            fes.append(after)
            r2_values.append(float(best_by_weight.mean()))
            archive_sizes.append(len(archive))

    if offspring_buffer:
        raise RuntimeError(
            "Incomplete MOEA/D generation at FE_MAX. "
            "Choose FE_MAX so that FE_MAX is divisible by POP_SIZE."
        )

    return {
        "pops": populations,  # needed by MGBM and ESC.
        "offs": offspring,    # needed by \(\epsilon\)SC.
        "fe": np.asarray(fes),
        "r2": np.asarray(r2_values),   # needed by your proposed rule and hindsight benchmark.
        "archive_size": np.asarray(archive_sizes),
        "ideal": ideal,
        "nadir": nadir,
    }

In [11]:
# Save each complete trajectory
def save_trajectory(name, m, run, seed, trajectory):
    path = TRAJECTORY_DIR / (
        f"{name}_m{m}_run{run:02d}_seed{seed}.npz"
    )

    np.savez_compressed(
        path,
        pops=np.stack(trajectory["pops"]),
        offs=np.stack(trajectory["offs"]),
        fe=trajectory["fe"],
        r2=trajectory["r2"],
        archive_size=trajectory["archive_size"],
        ideal=trajectory["ideal"],
        nadir=trajectory["nadir"],
    )

    return path

In [ ]:
# Loading an existing trajectory
def load_trajectory(path):
    data = np.load(path)

    return {
        "pops": list(data["pops"]),
        "offs": list(data["offs"]),
        "fe": data["fe"],
        "r2": data["r2"],
        "archive_size": data["archive_size"],
        "ideal": data["ideal"],
        "nadir": data["nadir"],
    }

## instead of 50, do not rerun 100,000 MOEA/D evaluations, simply load the trajectory and rerun:
# trajectory = load_trajectory(path)
# rows = summarize(name, m, run, seed, trajectory)

In [13]:
# MGBM
def dominated_count(target, reference):
    le = reference[:, None, :] <= target[None, :, :]
    lt = reference[:, None, :] < target[None, :, :]
    dominated = (le.all(axis=2) & lt.any(axis=2)).any(axis=0)
    return int(dominated.sum())


def stop_mgbm(populations):
    estimate = 1.0
    covariance = MGBM_R

    for i in range(1, len(populations)):
        previous = populations[i - 1]
        current = populations[i]

        mdr = (
            dominated_count(previous, current) / len(previous)
            - dominated_count(current, previous) / len(current)
        )

        gain = covariance / (covariance + MGBM_R)
        estimate = estimate + gain * (mdr - estimate)

        if estimate < MGBM_THRESHOLD:
            return i, True

        covariance = MGBM_R / (MGBM_R + covariance) * covariance

    return len(populations) - 1, False

In [14]:
# Entropy-based stopping criterion
def cell_counts(F, lower, upper):
    Z = (F - lower) / (upper - lower + 1e-12)
    indices = (Z * ESC_BINS).astype(int)
    base = ESC_BINS ** np.arange(F.shape[1])
    return Counter((indices @ base).tolist())


def entropy_measure(previous, current):
    X = np.vstack((previous, current))
    lower = X.min(axis=0)
    upper = X.max(axis=0)

    previous_cells = cell_counts(previous, lower, upper)
    current_cells = cell_counts(current, lower, upper)

    value = 0.0

    for cell, count in previous_cells.items():
        p = count / len(previous)
        q = current_cells.get(cell, 0) / len(current)

        if q > 0:
            value -= 0.5 * (p + q) * math.log(p / q)
        else:
            value -= p * math.log(p)

    for cell, count in current_cells.items():
        if cell not in previous_cells:
            q = count / len(current)
            value -= q * math.log(q)

    return value


def round_half_up(x):
    return float(
        Decimal(str(float(x))).quantize(
            Decimal(ESC_ROUND),
            ROUND_HALF_UP,
        )
    )


def stop_esc(populations):
    cumulative_mean = 0.0
    cumulative_square = 0.0

    mean_history = deque([0.0] * (ESC_WINDOW - 1), maxlen=ESC_WINDOW)
    variance_history = deque([0.0] * (ESC_WINDOW - 1), maxlen=ESC_WINDOW)

    for i in range(1, len(populations)):
        generation = i + 1
        value = entropy_measure(populations[i - 1], populations[i])

        cumulative_mean += value
        cumulative_square += value * value

        mean = cumulative_mean / (generation + 1)
        variance = cumulative_square / (generation + 1) - mean * mean

        mean_history.append(round_half_up(mean))
        variance_history.append(round_half_up(variance))

        if (
            generation > ESC_WINDOW
            and len(set(mean_history)) <= 1
            and len(set(variance_history)) <= 1
        ):
            return i, True

    return len(populations) - 1, False

In [15]:
# Epsilon-progress stopping criterion
def epsilon_archive_add(archive, F, problem):
    for f in F:
        solution = Solution(problem)
        solution.objectives[:] = f.tolist()
        archive.add(solution)


def stop_epsilon(initial_population, offspring, m):
    problem = PlatypusProblem(0, m)
    archive = EpsilonBoxArchive([EPSILON])

    epsilon_archive_add(archive, initial_population, problem)
    stagnation = 0

    for generation, Q in enumerate(offspring, start=1):
        before = archive.improvements
        epsilon_archive_add(archive, Q, problem)

        if archive.improvements > before:
            stagnation = 0
        else:
            stagnation += 1

        if stagnation >= EPS_STAGNATION:
            return generation, True

    return len(offspring), False

In [16]:
# Proposed cost-aware stopping criterion
def stop_proposed(r2_values):
    streak = 0
    generation_cost = C_FE * POP_SIZE

    for i in range(T_MIN - 1, len(r2_values)):
        prediction = fit_predict_next(r2_values[:i + 1])

        if prediction is None:
            streak = 0
            continue

        improvement = r2_values[i] - prediction

        if improvement <= generation_cost:
            streak += 1
        else:
            streak = 0

        if streak >= PATIENCE:
            return i, True, prediction, improvement

    return len(r2_values) - 1, False, np.nan, np.nan

In [17]:
# Hindsight best
def stop_hindsight(r2_values, fe):
    performance = r2_values + C_FE * fe
    return int(np.argmin(performance))

In [18]:
# Collect results
def summarize(name, m, run, seed, trajectory):
    r2_values = trajectory["r2"]
    fe = trajectory["fe"]
    archive_sizes = trajectory["archive_size"]

    proposed_idx, proposed_hit, prediction, improvement = stop_proposed(r2_values)
    mgbm_idx, mgbm_hit = stop_mgbm(trajectory["pops"])
    esc_idx, esc_hit = stop_esc(trajectory["pops"])
    epsilon_idx, epsilon_hit = stop_epsilon(
        trajectory["pops"][0],
        trajectory["offs"],
        m,
    )
    hindsight_idx = stop_hindsight(r2_values, fe)

    methods = [
        ("Proposed", proposed_idx, proposed_hit, prediction, improvement),
        ("MGBM", mgbm_idx, mgbm_hit, np.nan, np.nan),
        ("ESC", esc_idx, esc_hit, np.nan, np.nan),
        ("epsilonSC", epsilon_idx, epsilon_hit, np.nan, np.nan),
        ("Hindsight", hindsight_idx, True, np.nan, np.nan),
    ]

    decomposition = "Tchebycheff" if m == 2 else "PBI"
    rows = []

    for method, idx, triggered, pred, delta in methods:
        rows.append(
            {
                "problem": name.upper(),
                "n_obj": m,
                "run": run,
                "seed": seed,
                "algorithm": "MOEA/D",
                "decomposition": decomposition,
                "method": method,
                "triggered": triggered,
                "stop_gen": idx + 1,
                "stop_fe": int(fe[idx]),
                "archive_size": int(archive_sizes[idx]),
                "R2": float(r2_values[idx]),
                "evaluation_cost": float(C_FE * fe[idx]),
                "performance": float(r2_values[idx] + C_FE * fe[idx]),
                "pred_next_R2": pred,
                "pred_improvement": delta,
            }
        )

    return rows

In [ ]:
# Full experiment
def run_experiment():
    rows = []

    for name in PROBLEMS:
        for m in M_LIST:
            for run in range(N_RUNS):
                seed = SEED0 + run

                print(
                    f"{name.upper()} | m={m} | "
                    f"run={run + 1}/{N_RUNS}",
                    flush=True,
                )

                trajectory = run_trajectory(name, m, seed)
                # save_trajectory(name, m, run, seed, trajectory)
                rows.extend(summarize(name, m, run, seed, trajectory))

                pd.DataFrame(rows).to_csv(OUT, index=False)

    df = pd.DataFrame(rows)
    print(f"Saved {len(df)} rows to {OUT.resolve()}")
    return df

# Run
if __name__ == "__main__":
    results = run_experiment()